# Cell Tracking with VoxelMorph — Training on Colab

2D deformable registration for PhC-C2DH-U373 cell tracking using VoxelMorph.

**Before running:** Go to `Runtime > Change runtime type > GPU (T4)`

## 1. Setup

In [ ]:
# Clone the repo and install
!git clone https://github.com/abdullahh-sheikhh/voxelmorph.git
%cd voxelmorph
!git checkout dev
!pip install -e . -q
!pip install imagecodecs -q

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Download Dataset

In [ ]:
import os
from pathlib import Path

# Download PhC-C2DH-U373 dataset from Cell Tracking Challenge
!mkdir -p dataset

if not Path('dataset/train/01').exists():
    print('Downloading training data...')
    !wget -q https://data.celltrackingchallenge.net/training-datasets/PhC-C2DH-U373.zip -O dataset/train.zip
    !cd dataset && unzip -q train.zip -d train_raw && mv train_raw/PhC-C2DH-U373/* train/ && rm -rf train_raw train.zip
    # Rename to match expected structure (CTC uses different naming)
    !ls dataset/train/
else:
    print('Training data already exists')

if not Path('dataset/test/01').exists():
    print('Downloading test data...')
    !wget -q https://data.celltrackingchallenge.net/test-datasets/PhC-C2DH-U373.zip -O dataset/test.zip
    !cd dataset && unzip -q test.zip -d test_raw && mv test_raw/PhC-C2DH-U373/* test/ && rm -rf test_raw test.zip
    !ls dataset/test/
else:
    print('Test data already exists')

# Verify
!echo "Train:" && ls dataset/train/01/ | head -5 && echo "..." && ls dataset/train/01/ | wc -l
!echo "Test:"  && ls dataset/test/01/  | head -5 && echo "..." && ls dataset/test/01/  | wc -l

## 3. Train

In [ ]:
# Train — this should take a few minutes on a T4 GPU
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 \
    --batch-size 4 \
    --lr 1e-4 \
    --lambda 0.01 \
    --output-dir output \
    --save-every 25

## 4. Evaluate

In [ ]:
# Evaluate on training data (where GT masks exist) with Dice score
!python -m scripts.cell_tracking.evaluate \
    --model output/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --output-dir output/eval \
    --max-pairs 0

In [ ]:
# Display evaluation results
from IPython.display import Image, display
from pathlib import Path

eval_dir = Path('output/eval')
for img_path in sorted(eval_dir.glob('*.png'))[:5]:
    print(f'\n{img_path.name}')
    display(Image(filename=str(img_path), width=900))

## 5. Track Cells

In [ ]:
# Track cells across sequence 01 using the trained model
!python -m scripts.cell_tracking.track \
    --model output/best.pt \
    --data-dir dataset/train \
    --sequence 01 \
    --output-dir output/tracking \
    --viz-every 10

In [ ]:
# Display tracking results
from IPython.display import Image, display
from pathlib import Path

track_dir = Path('output/tracking')

# Show trajectory plot
print('Cell trajectories:')
display(Image(filename=str(track_dir / 'trajectories.png'), width=700))

# Show tracking overlays
for img_path in sorted(track_dir.glob('track_frame_*.png'))[:6]:
    print(f'\n{img_path.name}')
    display(Image(filename=str(img_path), width=900))

## 6. Model Variant Experiments

Train and compare different configurations from the paper (Table I):
- **VM-1**: MSE loss, direct displacement (baseline — already trained above)
- **VM-2**: NCC loss, direct displacement
- **VM-3**: MSE loss, diffeomorphic (int_steps=7)
- **VM-4**: NCC loss, diffeomorphic (int_steps=7)

In [ ]:
# VM-2: NCC loss, direct displacement
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss ncc --lambda 1.0 --int-steps 0 \
    --output-dir output/vm2_ncc \
    --save-every 50

In [ ]:
# VM-3: MSE loss, diffeomorphic (int_steps=7)
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss mse --lambda 0.01 --int-steps 7 \
    --output-dir output/vm3_diffeo \
    --save-every 50

In [ ]:
# VM-4: NCC loss, diffeomorphic (int_steps=7)
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --epochs 100 --batch-size 4 --lr 1e-4 \
    --loss ncc --lambda 1.0 --int-steps 7 \
    --output-dir output/vm4_diffeo_ncc \
    --save-every 50

In [ ]:
# Evaluate all variants with Dice
import subprocess

variants = {
    'VM-1 (MSE, direct)': ('output/best.pt', 0, 'output/eval_vm1'),
    'VM-2 (NCC, direct)': ('output/vm2_ncc/best.pt', 0, 'output/eval_vm2'),
    'VM-3 (MSE, diffeo)': ('output/vm3_diffeo/best.pt', 7, 'output/eval_vm3'),
    'VM-4 (NCC, diffeo)': ('output/vm4_diffeo_ncc/best.pt', 7, 'output/eval_vm4'),
}

for name, (model_path, int_steps, eval_dir) in variants.items():
    if not Path(model_path).exists():
        print(f'Skipping {name}: {model_path} not found')
        continue
    print(f'\n{"="*60}')
    print(f'Evaluating {name}...')
    print(f'{"="*60}')
    !python -m scripts.cell_tracking.evaluate \
        --model {model_path} \
        --data-dir dataset/train \
        --gt-dir dataset/train \
        --int-steps {int_steps} \
        --output-dir {eval_dir} \
        --max-pairs 0

In [ ]:
# Comparison table
import json
from IPython.display import display, Markdown

eval_dirs = {
    'VM-1 (MSE, direct)': 'output/eval_vm1',
    'VM-2 (NCC, direct)': 'output/eval_vm2',
    'VM-3 (MSE, diffeo)': 'output/eval_vm3',
    'VM-4 (NCC, diffeo)': 'output/eval_vm4',
}

rows = ['| Variant | MSE | Dice | Folding % | Runtime (s/pair) |',
        '|---------|-----|------|-----------|------------------|']

for name, eval_dir in eval_dirs.items():
    metrics_path = Path(eval_dir) / 'metrics.json'
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        m = json.load(f)
    dice = f"{m['dice_mean']:.4f} +/- {m['dice_std']:.4f}" if 'dice_mean' in m else 'N/A'
    rows.append(
        f"| {name} "
        f"| {m['mse_mean']:.6f} +/- {m['mse_std']:.6f} "
        f"| {dice} "
        f"| {m['folding_mean']:.2f}% "
        f"| {m['runtime_mean']:.4f} |"
    )

display(Markdown('\n'.join(rows)))

## 7. Download Results

In [ ]:
# Download all results (models, evaluations, tracking, experiments)
from google.colab import files

!zip -r results.zip output/best.pt output/final.pt output/config.json \
    output/eval/ output/tracking/ \
    output/vm2_ncc/ output/vm3_diffeo/ output/vm4_diffeo_ncc/ \
    output/eval_vm1/ output/eval_vm2/ output/eval_vm3/ output/eval_vm4/ \
    2>/dev/null; true
files.download('results.zip')